# Week 11 — Preprocessing ulasan SATUSEHAT Mobile

Notebook ini menjalankan cleaning dan pembagian data pada **dataset lengkap**.
Python tetap digunakan, tetapi langkahnya dapat dijalankan per sel di Jupyter.
Fungsi pengolahan berada di `src/preprocess.py` agar notebook dan CLI memakai
aturan yang sama dan tes tetap relevan.

Urutan: **Restart Kernel → Run All**. Pilih kernel environment proyek.
Pasang kebutuhan lewat terminal: `python -m pip install -r requirements-notebook.txt`.
Tidak ada API atau model LLM yang dipanggil dan tidak ada label manusia diisi.
Data bersih dan split yang sudah ada diperiksa, bukan ditimpa.

## 1. Lingkungan dan lokasi proyek

In [ ]:
from pathlib import Path
import csv, hashlib, io, json, sys
from collections import Counter
from html import escape
from IPython.display import display, HTML

# Berjalan dari akar repo maupun folder notebooks, tanpa path C: tertentu.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src/preprocess.py").is_file() and (p / "data").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Buka notebook dari checkout repository Web Mining.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src import preprocess, annotation

def read_csv(path):
    with Path(path).open(encoding="utf-8-sig", newline="") as stream:
        return list(csv.DictReader(stream))

def show_table(rows, columns=None, limit=10):
    rows = list(rows)
    if not rows:
        print("Tidak ada baris.")
        return
    columns = columns or list(rows[0])
    header = "".join(f"<th>{escape(str(c))}</th>" for c in columns)
    body = "".join("<tr>" + "".join(f"<td>{escape(str(r.get(c, '')))}</td>"
                   for c in columns) + "</tr>" for r in rows[:limit])
    display(HTML(f"<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>"))

print("Root:", ROOT)
print("Python:", sys.version.split()[0])

## 2. Baca dan periksa data asli

Satu baris mewakili satu ulasan. Rating adalah metadata, bukan label sentimen
aspek. Hash sumber dibandingkan dengan versi hasil preprocessing yang dicatat.
Jika sumber berubah, gunakan versi split baru yang disepakati tim.

In [ ]:
SOURCE = ROOT / "data/raw/com.telkom.tracencare_reviews.csv"
BASELINE = ROOT / "results/preprocessing/v1/summary.json"
baseline = json.loads(BASELINE.read_text(encoding="utf-8"))
assert preprocess.sha(SOURCE) == baseline["source_sha256"], "Dataset berubah; periksa versi."
raw = read_csv(SOURCE)
print("Jumlah ulasan:", len(raw))
print("Kolom:", list(raw[0]))
show_table(raw, ["at", "score", "content", "reviewCreatedVersion"], limit=5)

## 3. Cleaning dan dua versi teks

Aturan: sejak 1 Maret 2023; teks tidak kosong; spasi rapi; minimal tiga kata;
masking pola informasi pribadi; teks LLM mempertahankan kapital dan emoji;
teks model memakai huruf kecil, emoji Indonesia, dan pengurangan huruf berulang.
Tanpa stopword removal dan stemming. Normalisasi slang khusus B1 belum termasuk.

Duplikat disimpan yang paling awal untuk pool; clean tetap memuat duplikat
untuk analisis per versi. `in_pool=1` berarti masuk pembagian data. Regex
masking tidak menjamin semua informasi pribadi terdeteksi.

In [ ]:
# Menjalankan cleaning di memori; data asli tidak diedit.
cleaned, counts, masks = preprocess.clean_rows(raw)
show_table([{"Tahap": name, "Jumlah": count} for name, count in counts.items()])
print("Kecocokan pola masking:", masks)
assert counts == baseline["counts"], "Hasil berubah dari baseline; periksa versi kode."
show_table(cleaned, ["text_llm", "text_model", "n_words", "in_pool"], limit=5)

## 4. Pembagian data berstrata rating

Seed 42; 50 kalibrasi, 400 gold latih, 100 validasi, 300 uji, sisanya silver.
Stratifikasi rating tidak menyatakan distribusi sentimen/aspek. `silver`
berarti calon data untuk LLM, **belum label silver yang selesai**.

In [ ]:
split = preprocess.make_split(cleaned, seed=42)
lookup = {r["reviewId"]: r for r in cleaned}
split_counts = Counter(r["split"] for r in split)
show_table([{"Split": k, "Jumlah": v} for k, v in sorted(split_counts.items())])
ratings = []
for name in [n for n, _ in preprocess.SIZES] + ["silver"]:
    stars = Counter(lookup[r["reviewId"]]["score"] for r in split if r["split"] == name)
    ratings.append({"Split": name, **{str(s): stars[str(s)] for s in range(1, 6)}})
show_table(ratings)

## 5. Periksa ID, teks identik, dan keterulangan

In [ ]:
ids = [r["reviewId"] for r in split]
assert len(ids) == len(set(ids))
assert set(ids) == {r["reviewId"] for r in cleaned if r["in_pool"]}
for field in ["text_llm", "text_model"]:
    normalized = [" ".join(lookup[r["reviewId"]][field].lower().split()) for r in split]
    assert len(normalized) == len(set(normalized)), f"Teks identik pada {field}"
assert split == preprocess.make_split(list(reversed(cleaned)), seed=42)
assert preprocess.sha(SOURCE) == baseline["source_sha256"]
print("LULUS: ID unik, teks identik terpisah, split dapat diulang, sumber utuh.")
print("Batas: parafrasa/duplikat dekat belum diperiksa; split bukan temporal.")

## 6. Simpan keluaran atau verifikasi berkas yang sudah ada

Sel ini menulis hanya berkas yang belum ada. Bila berkas sudah ada namun
isinya berbeda, proses dihentikan sebelum menulis apa pun. Ini melindungi
split yang dipakai anotasi. Jangan menghapus/mengganti split tanpa kesepakatan.

In [ ]:
def csv_bytes(fields, rows):
    stream = io.StringIO(newline="")
    writer = csv.DictWriter(stream, fieldnames=fields)
    writer.writeheader()
    writer.writerows(rows)
    return stream.getvalue().encode("utf-8-sig")

CLEAN_PATH = ROOT / "data/processed/reviews_clean.csv"
SPLIT_PATH = ROOT / "data/annotations/split.csv"
payloads = {CLEAN_PATH: csv_bytes(preprocess.FIELDS, cleaned),
            SPLIT_PATH: csv_bytes(["reviewId", "split"], split)}
for path, payload in payloads.items():
    if path.exists() and path.read_bytes() != payload:
        raise ValueError(f"Keluaran berbeda: {path}; jangan menimpa versi anotasi.")
for path, payload in payloads.items():
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        with path.open("xb") as stream:
            stream.write(payload)
        print("Dibuat:", path.relative_to(ROOT))
    else:
        print("Terverifikasi tanpa ditimpa:", path.relative_to(ROOT))
assert preprocess.sha(CLEAN_PATH) == baseline["outputs_sha256"]["reviews_clean"]
assert preprocess.sha(SPLIT_PATH) == baseline["outputs_sha256"]["split"]
print("Hash clean dan split sesuai hasil yang dicatat.")

## 7. Kesimpulan dan langkah berikutnya

Dataset awal 39.995 ulasan → 32.843 lolos filter dasar → **32.035 pool unik**.
Ada 808 duplikat setelah filter panjang; angka duplikat eksplorasi awal memakai
populasi berbeda. Calon silver berjumlah 31.185.

Lanjutkan `02_kalibrasi_anotasi.ipynb` untuk lembar kosong dan audit setelah
diisi manusia. Kappa, gold final, training, dan skor ABSA belum tersedia.
Pengujian perilaku kode: `python -m unittest discover -s tests -v`.